# Supermarket Sales Analysis

**Data Analytics Project using Python, Pandas and Matplotlib**

## Objectives
- Load and inspect supermarket sales data
- Clean and validate the dataset
- Perform exploratory data analysis (EDA)
- Calculate important KPIs
- Answer business questions
- Visualize sales, products, branches, customers and payments
- Generate actionable business recommendations


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load Dataset

In [ ]:
# Update this path if the notebook is moved
file_path = r"SUPER MARKET DATA.xlsx"

df = pd.read_excel(file_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()


## 3. Inspect the Data

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

display(df.describe(include="all").T)


## 4. Clean Column Names

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
      .str.replace("-", "_", regex=False)
)

df.columns.tolist()


## 5. Data Cleaning & Validation

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Convert numeric columns where available
numeric_candidates = [
    "unit_price", "quantity", "tax", "total", "cogs",
    "gross_margin_percentage", "gross_income", "rating", "sales"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate Sales = Quantity × Unit Price when possible
if {"quantity", "unit_price"}.issubset(df.columns):
    df["sales_calculated"] = df["quantity"] * df["unit_price"]

print("Rows after duplicate removal:", len(df))
print("\nMissing values:")
display(df.isnull().sum())

if {"sales_calculated", "sales"}.issubset(df.columns):
    df["sales_difference"] = (df["sales"] - df["sales_calculated"]).abs()
    print("Maximum Sales calculation difference:",
          df["sales_difference"].max())

if "rating" in df.columns:
    print("Ratings outside 0–5:",
          ((df["rating"] < 0) | (df["rating"] > 5)).sum())


## 6. KPI Calculations

In [ ]:
total_sales = df["sales"].sum() if "sales" in df.columns else np.nan
total_transactions = len(df)
total_quantity = df["quantity"].sum() if "quantity" in df.columns else np.nan
average_transaction = df["sales"].mean() if "sales" in df.columns else np.nan
average_rating = df["rating"].mean() if "rating" in df.columns else np.nan

kpis = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Transactions",
        "Total Quantity",
        "Average Transaction Value",
        "Average Customer Rating"
    ],
    "Value": [
        total_sales,
        total_transactions,
        total_quantity,
        average_transaction,
        average_rating
    ]
})

display(kpis)


## 7. Exploratory Data Analysis — Sales by Product

In [ ]:
product_col = "product_line" if "product_line" in df.columns else "product"

sales_by_product = (
    df.groupby(product_col)["sales"]
      .sum()
      .sort_values(ascending=False)
)

display(sales_by_product.to_frame("Total Sales"))

plt.figure(figsize=(10, 5))
sales_by_product.plot(kind="bar")
plt.title("Sales by Product / Product Line")
plt.xlabel("Product")
plt.ylabel("Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 8. Sales by Branch

In [ ]:
branch_sales = df.groupby("branch")["sales"].sum().sort_values(ascending=False)
display(branch_sales.to_frame("Total Sales"))

plt.figure(figsize=(7, 5))
branch_sales.plot(kind="bar")
plt.title("Sales by Branch")
plt.xlabel("Branch")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Sales by City

In [ ]:
if "city" in df.columns:
    city_sales = df.groupby("city")["sales"].sum().sort_values(ascending=False)
    display(city_sales.to_frame("Total Sales"))


## 10. Payment Method Analysis

In [ ]:
payment_counts = df["payment_method"].value_counts()

display(payment_counts.to_frame("Transactions"))

plt.figure(figsize=(7, 5))
payment_counts.plot(kind="bar")
plt.title("Payment Method Usage")
plt.xlabel("Payment Method")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Member vs Normal Customers

In [ ]:
customer_avg = (
    df.groupby("customer_type")["sales"]
      .mean()
      .sort_values(ascending=False)
)

display(customer_avg.to_frame("Average Transaction"))

plt.figure(figsize=(7, 5))
customer_avg.plot(kind="bar")
plt.title("Average Transaction by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Average Transaction")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. Customer Rating Analysis

In [ ]:
if "rating" in df.columns:
    rating_by_branch = df.groupby("branch")["rating"].mean().sort_values(ascending=False)
    print("Overall Average Rating:", round(df["rating"].mean(), 2))
    display(rating_by_branch.to_frame("Average Rating"))

    plt.figure(figsize=(7, 5))
    rating_by_branch.plot(kind="bar")
    plt.title("Average Customer Rating by Branch")
    plt.xlabel("Branch")
    plt.ylabel("Average Rating")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 13. Business Questions & Answers

In [ ]:
# Highest-selling product
top_product = sales_by_product.idxmax()
top_product_sales = sales_by_product.max()

# Best branch
top_branch = branch_sales.idxmax()
top_branch_sales = branch_sales.max()

# Highest-selling category/product line
top_category = sales_by_product.idxmax()
top_category_sales = sales_by_product.max()

# Most popular payment method
top_payment = payment_counts.idxmax()
top_payment_count = payment_counts.max()

print(f"1. Highest-selling product/product line: {top_product} — ₹{top_product_sales:,.2f}")
print(f"2. Best-performing branch: {top_branch} — ₹{top_branch_sales:,.2f}")
print(f"3. Highest-selling category/product line: {top_category} — ₹{top_category_sales:,.2f}")
print(f"4. Most popular payment method: {top_payment} — {top_payment_count} transactions")

if "customer_type" in df.columns:
    member_avg = df.loc[df["customer_type"].eq("Member"), "sales"].mean()
    normal_avg = df.loc[df["customer_type"].eq("Normal"), "sales"].mean()
    print(f"5. Member average transaction: ₹{member_avg:,.2f}")
    print(f"   Normal average transaction: ₹{normal_avg:,.2f}")

print(f"6. Average customer rating: {average_rating:.2f}/5")


## 14. Key Insights

Based on the analysis, identify:
- The product/product line contributing the most revenue
- The branch contributing the most revenue
- The category/product line with the strongest sales
- The dominant payment method
- Differences between Member and Normal customers
- Branch-level customer satisfaction patterns

These findings can be used to support inventory planning, branch performance reviews, payment strategy and customer-service improvements.


## 15. Business Recommendations

1. Maintain sufficient inventory for high-selling products and categories.
2. Investigate the operational factors contributing to the strongest branch performance.
3. Continue supporting widely used digital payment methods such as UPI.
4. Monitor customer ratings by branch and product category to identify service opportunities.
5. Compare Member and Normal purchasing behavior before designing membership campaigns.
6. Use sales trends to improve inventory allocation and promotional planning.


## 16. Export Cleaned Dataset

The cleaned dataset can be exported for SQL and Power BI analysis.


In [ ]:
output_file = "supermarket_sales_cleaned.csv"
df.to_csv(output_file, index=False)
print(f"Saved: {output_file}")
